# Fine-tune VietOCR Address-only from VietOCR v2

This notebook trains a dedicated recognition model for the CCCD `address` field.

Prepare these files in Google Drive `/content/drive/MyDrive/`:
- `vietocr_address_only.zip` generated from this repo.
- `vietocr_cccd_v2.pth` copied from `weights/vietocr_cccd_v2.pth`.

Main outputs:
- `vietocr_cccd_address_v1.pth`
- `vietocr_address_v1_checkpoint.ckpt`
- `vietocr_address_v1_eval.json`

Recommended integration: use this model only for `address`; keep VietOCR v2 for other fields.


## 1. Setup


In [1]:
!pip install -q vietocr lmdb imgaug
!pip install -q 'Pillow>=10.0,<11.0' --force-reinstall


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.0/948.0 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.9/133.9 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 84.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.3/338.3 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 92.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
vietocr 0.3.13 requires pillow==10.2.0, but you have pillow 10.4.0 which is incompatible.


In [2]:
import numpy as np
if not hasattr(np, 'sctypes'):
    np.sctypes = {
        'int': [np.int8, np.int16, np.int32, np.int64],
        'uint': [np.uint8, np.uint16, np.uint32, np.uint64],
        'float': [np.float16, np.float32, np.float64],
        'complex': [np.complex64, np.complex128],
        'others': [bool, object, bytes, str, np.void],
    }

import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


PyTorch: 2.11.0+cu128
CUDA: True
GPU: Tesla T4


## 2. Mount Drive and extract data


In [3]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [4]:
from pathlib import Path
import zipfile, shutil, json, os, random, re, unicodedata

ZIP_PATH = Path('/content/drive/MyDrive/vietocr_address_only.zip')
BASE_WEIGHTS = Path('/content/drive/MyDrive/vietocr_cccd_v2.pth')
WORK_DIR = Path('/content/vietocr_address_v1')
DRIVE_OUT = Path('/content/drive/MyDrive/vietocr_address_v1')
DRIVE_CKPT = DRIVE_OUT / 'vietocr_address_v1_checkpoint.ckpt'

WORK_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_OUT.mkdir(parents=True, exist_ok=True)

assert ZIP_PATH.exists(), f'Missing {ZIP_PATH}'
assert BASE_WEIGHTS.exists(), f'Missing {BASE_WEIGHTS}; upload weights/vietocr_cccd_v2.pth to Google Drive first.'

if not (WORK_DIR / 'train_annotation.txt').exists():
    print('Extracting dataset...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(WORK_DIR)
else:
    print('Dataset already extracted.')

for name in ['train_annotation.txt', 'val_annotation.txt', 'test_annotation.txt', 'dataset_meta.json']:
    assert (WORK_DIR / name).exists(), f'Missing {name}'

meta = json.loads((WORK_DIR / 'dataset_meta.json').read_text(encoding='utf-8'))
print(json.dumps(meta, ensure_ascii=False, indent=2))


Extracting dataset...
{
  "name": "cccd_vietocr_address_only",
  "source_dir": "data/processed/ocr_address_origin_clean",
  "field": "address",
  "label_format": "relative_image_path<TAB>text",
  "splits": {
    "train": 1493,
    "val": 178,
    "test": 190
  },
  "normalization_examples": [
    {
      "split": "train",
      "path": "crops/address/image504_jpg.rf.c73f1edbb81d4559b250bf54fe3cae36_4861.jpg",
      "before": "Thôn 9, Vū Đoài, Vũ Thư, Thái Bình",
      "after": "Thôn 9, Vũ Đoài, Vũ Thư, Thái Bình"
    },
    {
      "split": "train",
      "path": "crops/address/image506_jpg.rf.c1c83e7316369adcb2be342ea07040c8_4058.jpg",
      "before": "Thôn 10, Xā Long Hà, Bù Gia Mập, Bình Phước",
      "after": "Thôn 10, Xã Long Hà, Bù Gia Mập, Bình Phước"
    },
    {
      "split": "train",
      "path": "crops/address/image54_jpg.rf.c4c6e25a7384f4f627ca4f1d52624c70_4134.jpg",
      "before": "Thôn Vü Xuyên, Yên Dương, Ý Yên, Nam Định",
      "after": "Thôn Vũ Xuyên, Yên Dương, Ý Y

## 3. Validate data


In [5]:
from PIL import Image
from statistics import mean

def read_ann(path):
    rows = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rel, text = line.split('	', 1)
            rows.append((rel, text))
    return rows

train_rows = read_ann(WORK_DIR / 'train_annotation.txt')
val_rows = read_ann(WORK_DIR / 'val_annotation.txt')
test_rows = read_ann(WORK_DIR / 'test_annotation.txt')

for name, rows in [('train', train_rows), ('val', val_rows), ('test', test_rows)]:
    missing = [rel for rel, _ in rows if not (WORK_DIR / rel).exists()]
    lengths = [len(text) for _, text in rows]
    print(name, 'rows=', len(rows), 'missing=', len(missing), 'avg/max length=', round(mean(lengths), 1), max(lengths))
    if missing:
        raise FileNotFoundError(missing[:5])

sizes = []
for rel, _ in train_rows[:300]:
    with Image.open(WORK_DIR / rel) as im:
        sizes.append((im.width, im.height, round(im.width / max(im.height, 1), 2)))
print('sample width min/avg/max:', min(w for w,_,_ in sizes), round(mean(w for w,_,_ in sizes),1), max(w for w,_,_ in sizes))
print('sample height min/avg/max:', min(h for _,h,_ in sizes), round(mean(h for _,h,_ in sizes),1), max(h for _,h,_ in sizes))
print('sample aspect min/avg/max:', min(r for _,_,r in sizes), round(mean(r for _,_,r in sizes),2), max(r for _,_,r in sizes))
print('samples:')
for rel, text in train_rows[:5]:
    print(rel, '=>', text)


train rows= 1493 missing= 0 avg/max length= 40.6 70
val rows= 178 missing= 0 avg/max length= 41.0 59
test rows= 190 missing= 0 avg/max length= 40.3 66
sample width min/avg/max: 344 503.0 691
sample height min/avg/max: 64 99.3 149
sample aspect min/avg/max: 3.31 5.17 8.28
samples:
crops/address/image100_jpg.rf.45b4e5087f8c5394faa21b9f2eee71f5_491.jpg => Ấp Hiệp Hòa Vĩnh Bình Bắc Vĩnh Thuận Kiên Giang
crops/address/image101_jpg.rf.2993b1c940ca1d00cbdcf491eaeb830c_4793.jpg => phú hòa tân hội tân hiệp kiên giang
crops/address/image101_jpg.rf.29d18c77b884f46fef60f564f2c919e4_4828.jpg => phú hòa tân hội tân hiệp kiên giang
crops/address/image101_jpg.rf.419c00cc3bf2286a381829c3f1317cb9_138.jpg => Phú Hòa Tân Hội Tân Hiệp Kiên Giang
crops/address/image101_jpg.rf.6e9583746570ab9e8e7ace813c69c228_11666.jpg => phú hòa tân hội tân hiệp kiên giang


## 4. Train config


In [6]:
import yaml

ITERS = 10000
MAX_LR = 1e-5
BATCH_SIZE = 8
IMAGE_HEIGHT = 48
IMAGE_MAX_WIDTH = 1024
VALID_EVERY = 1000
PRINT_EVERY = 100

VOCAB = 'aA\xe0\xc0\u1ea3\u1ea2\xe3\xc3\xe1\xc1\u1ea1\u1ea0\u0103\u0102\u1eb1\u1eb0\u1eb3\u1eb2\u1eb5\u1eb4\u1eaf\u1eae\u1eb7\u1eb6\xe2\xc2\u1ea7\u1ea6\u1ea9\u1ea8\u1eab\u1eaa\u1ea5\u1ea4\u1ead\u1eacbBcCdD\u0111\u0110eE\xe8\xc8\u1ebb\u1eba\u1ebd\u1ebc\xe9\xc9\u1eb9\u1eb8\xea\xca\u1ec1\u1ec0\u1ec3\u1ec2\u1ec5\u1ec4\u1ebf\u1ebe\u1ec7\u1ec6fFgGhHiI\xec\xcc\u1ec9\u1ec8\u0129\u0128\xed\xcd\u1ecb\u1ecajJkKlLmMnNoO\xf2\xd2\u1ecf\u1ece\xf5\xd5\xf3\xd3\u1ecd\u1ecc\xf4\xd4\u1ed3\u1ed2\u1ed5\u1ed4\u1ed7\u1ed6\u1ed1\u1ed0\u1ed9\u1ed8\u01a1\u01a0\u1edd\u1edc\u1edf\u1ede\u1ee1\u1ee0\u1edb\u1eda\u1ee3\u1ee2pPqQrRsStTuU\xf9\xd9\u1ee7\u1ee6\u0169\u0168\xfa\xda\u1ee5\u1ee4\u01b0\u01af\u1eeb\u1eea\u1eed\u1eec\u1eef\u1eee\u1ee9\u1ee8\u1ef1\u1ef0vVwWxXyY\u1ef3\u1ef2\u1ef7\u1ef6\u1ef9\u1ef8\xfd\xdd\u1ef5\u1ef4zZ0123456789!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~ '

config_dict = {
    'vocab': VOCAB,
    'backbone': 'vgg19_bn',
    'cnn': {'ss': [[2,2],[2,2],[2,1],[2,1],[1,1]], 'ks': [[2,2],[2,2],[2,1],[2,1],[1,1]], 'hidden': 256, 'dropout': 0.5},
    'transformer': {'d_model': 256, 'nhead': 8, 'num_encoder_layers': 6, 'num_decoder_layers': 6, 'dim_feedforward': 2048, 'max_seq_length': 1024, 'pos_dropout': 0.1, 'trans_dropout': 0.1},
    'seq_modeling': 'transformer',
    'pretrain': 'https://vocr.vn/data/vietocr/vgg_transformer.pth',
    'weights': '',
    'device': 'cuda:0' if torch.cuda.is_available() else 'cpu',
    'quiet': True,
    'dataset': {
        'name': 'cccd_address_only_v1',
        'data_root': str(WORK_DIR),
        'train_annotation': 'train_annotation.txt',
        'valid_annotation': 'val_annotation.txt',
        'image_height': IMAGE_HEIGHT,
        'image_min_width': 48,
        'image_max_width': IMAGE_MAX_WIDTH,
    },
    'dataloader': {'num_workers': 2, 'pin_memory': True},
    'predictor': {'beamsearch': False},
    'trainer': {
        'batch_size': BATCH_SIZE,
        'iters': ITERS,
        'print_every': PRINT_EVERY,
        'valid_every': VALID_EVERY,
        'checkpoint': str(WORK_DIR / 'checkpoint_address_v1.ckpt'),
        'export': str(WORK_DIR / 'vietocr_cccd_address_v1.pth'),
        'metrics': 1000,
        'log': str(WORK_DIR / 'train_log_address_v1'),
    },
    'optimizer': {'max_lr': MAX_LR, 'pct_start': 0.05},
    'aug': {'image_aug': True, 'masked_language_model': True},
}

CONFIG_PATH = WORK_DIR / 'config_address_v1.yml'
CONFIG_PATH.write_text(yaml.dump(config_dict, allow_unicode=True), encoding='utf-8')
print(CONFIG_PATH.read_text(encoding='utf-8'))


aug:
  image_aug: true
  masked_language_model: true
backbone: vgg19_bn
cnn:
  dropout: 0.5
  hidden: 256
  ks:
  - - 2
    - 2
  - - 2
    - 2
  - - 2
    - 1
  - - 2
    - 1
  - - 1
    - 1
  ss:
  - - 2
    - 2
  - - 2
    - 2
  - - 2
    - 1
  - - 2
    - 1
  - - 1
    - 1
dataloader:
  num_workers: 2
  pin_memory: true
dataset:
  data_root: /content/vietocr_address_v1
  image_height: 48
  image_max_width: 1024
  image_min_width: 48
  name: cccd_address_only_v1
  train_annotation: train_annotation.txt
  valid_annotation: val_annotation.txt
device: cuda:0
optimizer:
  max_lr: 1.0e-05
  pct_start: 0.05
predictor:
  beamsearch: false
pretrain: https://vocr.vn/data/vietocr/vgg_transformer.pth
quiet: true
seq_modeling: transformer
trainer:
  batch_size: 8
  checkpoint: /content/vietocr_address_v1/checkpoint_address_v1.ckpt
  export: /content/vietocr_address_v1/vietocr_cccd_address_v1.pth
  iters: 10000
  log: /content/vietocr_address_v1/train_log_address_v1
  metrics: 1000
  print_every

## 5. Train from VietOCR v2


In [7]:
from vietocr.tool.config import Cfg
from vietocr.model.trainer import Trainer

config = Cfg.load_config_from_file(str(CONFIG_PATH))
trainer = Trainer(config, pretrained=True)
print('Loading base VietOCR v2 weights:', BASE_WEIGHTS)
trainer.load_weights(str(BASE_WEIGHTS))

LOCAL_CKPT = Path(config['trainer']['checkpoint'])
if DRIVE_CKPT.exists():
    print('Found checkpoint on Drive, resuming:', DRIVE_CKPT)
    shutil.copy2(DRIVE_CKPT, LOCAL_CKPT)
    trainer.load_checkpoint(str(LOCAL_CKPT))
else:
    print('No address checkpoint found; start fine-tuning from VietOCR v2 weights.')

print('Device:', config['device'])
print('Train samples:', len(train_rows), 'Val samples:', len(val_rows))


Downloading: "https://download.pytorch.org/models/vgg19_bn-c79401a0.pth" to /root/.cache/torch/hub/checkpoints/vgg19_bn-c79401a0.pth


100%|██████████| 548M/548M [00:03<00:00, 189MB/s]
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:144: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.encoder = TransformerEncoder(
18533it [00:00, 27291.46it/s]
Create train_cccd_address_only_v1: 100%|███████████████████████| 1493/1493 [00:01<00:00, 974.03it/s]


Created dataset with 1492 samples


Create valid_cccd_address_only_v1: 100%|████████████████████████| 178/178 [00:00<00:00, 1932.19it/s]

Created dataset with 177 samples



valid_cccd_address_only_v1 build cluster: 100%|████████████████| 177/177 [00:00<00:00, 87525.56it/s]


Loading base VietOCR v2 weights: /content/drive/MyDrive/vietocr_cccd_v2.pth
No address checkpoint found; start fine-tuning from VietOCR v2 weights.
Device: cuda:0
Train samples: 1493 Val samples: 178


In [8]:
import threading, time

stop_auto_save = False

def auto_save_checkpoint(interval_sec=600):
    while not stop_auto_save:
        time.sleep(interval_sec)
        local_ckpt = Path(config['trainer']['checkpoint'])
        if local_ckpt.exists():
            try:
                shutil.copy2(local_ckpt, DRIVE_CKPT)
                print()
                print(f'[Auto-save] checkpoint copied to Drive at {time.strftime("%H:%M:%S")}')
            except Exception as exc:
                print()
                print('[Auto-save] failed:', exc)

saver = threading.Thread(target=auto_save_checkpoint, daemon=True)
saver.start()

trainer.train()
stop_auto_save = True
trainer.save_weights(config['trainer']['export'])
trainer.save_checkpoint(config['trainer']['checkpoint'])
shutil.copy2(config['trainer']['checkpoint'], DRIVE_CKPT)
print('Saved weights:', config['trainer']['export'])
print('Saved checkpoint to Drive:', DRIVE_CKPT)


iter: 000100 - train loss: 1.373 - lr: 1.32e-06 - load time: 0.18 - gpu time: 16.53
iter: 000200 - train loss: 1.290 - lr: 3.73e-06 - load time: 0.20 - gpu time: 15.11
iter: 000300 - train loss: 1.194 - lr: 6.70e-06 - load time: 0.08 - gpu time: 14.73
iter: 000400 - train loss: 1.095 - lr: 9.10e-06 - load time: 0.21 - gpu time: 15.22
iter: 000500 - train loss: 1.075 - lr: 1.00e-05 - load time: 0.06 - gpu time: 15.03
iter: 000600 - train loss: 0.996 - lr: 1.00e-05 - load time: 0.20 - gpu time: 15.55
iter: 000700 - train loss: 0.999 - lr: 9.99e-06 - load time: 0.06 - gpu time: 15.45
iter: 000800 - train loss: 0.970 - lr: 9.98e-06 - load time: 0.21 - gpu time: 15.46
iter: 000900 - train loss: 0.991 - lr: 9.96e-06 - load time: 0.19 - gpu time: 15.23
iter: 001000 - train loss: 0.954 - lr: 9.93e-06 - load time: 0.05 - gpu time: 15.31
iter: 001000 - valid loss: 0.873 - acc full seq: 0.2292 - acc per char: 0.4576
iter: 001100 - train loss: 0.940 - lr: 9.90e-06 - load time: 0.20 - gpu time: 15.

## 6. Eval val/test


In [10]:
from vietocr.tool.predictor import Predictor
from PIL import Image
import time

# The low-level vietocr.tool.translate.translate() expects a preprocessed tensor
# in newer VietOCR versions. Predictor.predict() accepts PIL images and handles preprocessing.
eval_config = dict(config)
eval_config['weights'] = config['trainer']['export']
eval_config['predictor']['beamsearch'] = False
predictor = Predictor(eval_config)

def levenshtein(a, b):
    if a == b:
        return 0
    if not a:
        return len(b)
    if not b:
        return len(a)
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]

def eval_rows(rows, name):
    exact = 0
    cer_sum = 0.0
    norm_edit_sum = 0.0
    details = []
    start = time.time()
    for rel, gt in rows:
        img = Image.open(WORK_DIR / rel).convert('RGB')
        pred, prob = predictor.predict(img, return_prob=True)
        pred = (pred or '').strip()
        dist = levenshtein(gt, pred)
        denom = max(len(gt), 1)
        cer = dist / denom
        norm_edit = 1 - dist / max(len(gt), len(pred), 1)
        exact += int(pred == gt)
        cer_sum += cer
        norm_edit_sum += norm_edit
        if len(details) < 30 and pred != gt:
            details.append({'path': rel, 'gt': gt, 'pred': pred, 'cer': cer, 'norm_edit': norm_edit, 'prob': float(prob) if prob is not None else None})
    elapsed = time.time() - start
    result = {
        'split': name,
        'n': len(rows),
        'exact_acc': exact / max(len(rows), 1),
        'cer': cer_sum / max(len(rows), 1),
        'norm_edit': norm_edit_sum / max(len(rows), 1),
        'fps': len(rows) / max(elapsed, 1e-6),
        'errors_sample': details,
    }
    print(name, json.dumps({k:v for k,v in result.items() if k != 'errors_sample'}, ensure_ascii=False, indent=2))
    return result

val_result = eval_rows(val_rows, 'val')
test_result = eval_rows(test_rows, 'test')
eval_result = {'val': val_result, 'test': test_result, 'config': {'iters': ITERS, 'max_lr': MAX_LR, 'batch_size': BATCH_SIZE, 'base_weights': str(BASE_WEIGHTS)}}
EVAL_PATH = WORK_DIR / 'vietocr_address_v1_eval.json'
EVAL_PATH.write_text(json.dumps(eval_result, ensure_ascii=False, indent=2), encoding='utf-8')
print('Eval saved:', EVAL_PATH)
print('Sample errors:')
for row in test_result['errors_sample'][:10]:
    print('GT  :', row['gt'])
    print('PRED:', row['pred'])
    print('CER :', round(row['cer'], 3), 'NED:', round(row['norm_edit'], 3))
    print()


val {
  "split": "val",
  "n": 178,
  "exact_acc": 0.3202247191011236,
  "cer": 0.15714966936445193,
  "norm_edit": 0.8547445236449968,
  "fps": 3.5400683949574243
}
test {
  "split": "test",
  "n": 190,
  "exact_acc": 0.2631578947368421,
  "cer": 0.21420343874528974,
  "norm_edit": 0.8004522435735348,
  "fps": 3.6012025281062305
}
Eval saved: /content/vietocr_address_v1/vietocr_address_v1_eval.json
Sample errors:
GT  : Mỹ Hạnh, Chánh An, Mang Thít, Vĩnh Long
PRED: 10 chánh an mang thít vĩnh long
CER : 0.41 NED: 0.59

GT  : 153 tổ 5 ấp lộ 25 bầu hàm 2 thống nhất đồng nai
PRED: 153 Tổ 5, Ấp Lộ 25 Bàu Hàm 2, Thống Nhất, Đồng Nai
CER : 0.277 NED: 0.74

GT  : 153, Tổ 5, Ấp Lô 25, Bàn Hàm 2, Thống Nhất, Đồng Nai
PRED: 363 tổ 5, ấp lộ 25 bàu hàm 2, thống nhai đồng ninh
CER : 0.404 NED: 0.596

GT  : ấp 5 thanh lợi tháp mười đồng tháp
PRED: Thanh Lei, Tháp Mười, Đồng Tháp
CER : 0.412 NED: 0.588

GT  : Ấp 5 Thanh Lợi, Tháp Mười, Đồng Tháp
PRED: ấn giáp thanh lợi tháp mười đồng tháp
CER : 0.389 

## 7. Copy artifacts to Drive


In [11]:
outputs = [
    Path(config['trainer']['export']),
    Path(config['trainer']['checkpoint']),
    CONFIG_PATH,
    EVAL_PATH,
]
for src in outputs:
    dst = DRIVE_OUT / src.name
    shutil.copy2(src, dst)
    print('Copied:', dst)

print()
print('Download this model into the project if metrics are good:')
print('  weights/vietocr_cccd_address_v1.pth')
print()
print('Do not replace weights/vietocr_cccd_v2.pth; integrate this only for field address.')


Copied: /content/drive/MyDrive/vietocr_address_v1/vietocr_cccd_address_v1.pth
Copied: /content/drive/MyDrive/vietocr_address_v1/checkpoint_address_v1.ckpt
Copied: /content/drive/MyDrive/vietocr_address_v1/config_address_v1.yml
Copied: /content/drive/MyDrive/vietocr_address_v1/vietocr_address_v1_eval.json

Download this model into the project if metrics are good:
  weights/vietocr_cccd_address_v1.pth

Do not replace weights/vietocr_cccd_v2.pth; integrate this only for field address.
